# Task 3: AQE Experiment (Queries 1, 6)

Compare the same Gold-table queries with Adaptive Query Execution (AQE) disabled and enabled.

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-AQE-Queries1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

QUERY_NAMES = (
    'query_01_monthly_zone_demand',
    'query_02_average_trip_distance_by_weather',
    'query_03_air_quality_vs_demand',
    'query_04_zone_weather_demand_variation',
    'query_05_peak_travel_hours',
    'query_06_monthly_taxi_demand_trend',
)
REPETITIONS = 9
AQE_DISABLED = 'false'
AQE_ENABLED = 'true'

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 11:14:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.read.format("delta") \
    .load("s3a://delta-data/silver/silver_weather_conditions") \
    .createOrReplaceTempView("weather_conditions")

26/09/20 11:15:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Measure AQE disabled and enabled

Each mode receives one warm-up followed by three timed runs.

In [3]:
def measure_aqe_mode(enabled):
    spark.catalog.clearCache()
    spark.conf.set('spark.sql.adaptive.enabled', enabled)
    register_integrated_taxi_trips(spark)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

aqe_disabled_results = measure_aqe_mode(AQE_DISABLED)
aqe_enabled_results = measure_aqe_mode(AQE_ENABLED)

for query_name in QUERY_NAMES:
    assert rows_match(
        aqe_disabled_results[query_name]['rows'],
        aqe_enabled_results[query_name]['rows'],
    ), f'AQE results differ for {query_name}.'

print('AQE-enabled and AQE-disabled results match within float tolerance.')

26/09/20 11:15:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AQE-enabled and AQE-disabled results match within float tolerance.


## AQE-disabled physical plans

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_DISABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-disabled plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

AQE-disabled plan: query_01_monthly_zone_demand
== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#23561, pickup_zone_name#23568, date#23602]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pickup_date#23561, pickup_zone_name#23568, date#23602]

(3) Project [codegen id : 1]
Output [2]: [pickup_zone_name#23568, substring(pickup_date#23561, 1, 7) AS pickup_month#23603]
Input [3]: [pickup_date#23561, pickup_zone_name#23568, date#23602]

(4) HashAggregate [codegen id : 1]
Input [2]: [pickup_zone_name#23568, pickup_month#23603]
Keys [2]: [pickup_zone_name#23568, pickup_month#23603]
Functions [1]: [partial_coun

## AQE-enabled physical plans

AEQ enabled -> `AdaptiveSparkPlan` and changed shuffles or joins.

In [5]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_ENABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-enabled plan: {query_name}')
    query = execute_registered_query(spark, query_name)
    query.collect()
    query.explain('formatted')

AQE-enabled plan: query_01_monthly_zone_demand


== Physical Plan ==
AdaptiveSparkPlan (20)
+- == Final Plan ==
   ResultQueryStage (13)
   +- * Sort (12)
      +- AQEShuffleRead (11)
         +- ShuffleQueryStage (10), Statistics(sizeInBytes=45.5 KiB, rowCount=790)
            +- Exchange (9)
               +- * HashAggregate (8)
                  +- AQEShuffleRead (7)
                     +- ShuffleQueryStage (6), Statistics(sizeInBytes=533.0 KiB, rowCount=9.28E+3)
                        +- Exchange (5)
                           +- * HashAggregate (4)
                              +- * Project (3)
                                 +- * ColumnarToRow (2)
                                    +- Scan parquet  (1)
+- == Initial Plan ==
   Sort (19)
   +- Exchange (18)
      +- HashAggregate (17)
         +- Exchange (16)
            +- HashAggregate (15)
               +- Project (14)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#24708, pickup_zone_name#24715, date#24749]
Batched: true
Location: Pr

== Physical Plan ==
AdaptiveSparkPlan (33)
+- == Final Plan ==
   ResultQueryStage (21)
   +- * Sort (20)
      +- AQEShuffleRead (19)
         +- ShuffleQueryStage (18), Statistics(sizeInBytes=496.0 B, rowCount=14)
            +- Exchange (17)
               +- * HashAggregate (16)
                  +- AQEShuffleRead (15)
                     +- ShuffleQueryStage (14), Statistics(sizeInBytes=4.7 KiB, rowCount=113)
                        +- Exchange (13)
                           +- * HashAggregate (12)
                              +- * Project (11)
                                 +- * BroadcastHashJoin Inner BuildRight (10)
                                    :- * Project (4)
                                    :  +- * Filter (3)
                                    :     +- * ColumnarToRow (2)
                                    :        +- Scan parquet  (1)
                                    +- BroadcastQueryStage (9), Statistics(sizeInBytes=1024.2 KiB, rowCount=27)
            

== Physical Plan ==
AdaptiveSparkPlan (34)
+- == Final Plan ==
   ResultQueryStage (21)
   +- * Sort (20)
      +- AQEShuffleRead (19)
         +- ShuffleQueryStage (18), Statistics(sizeInBytes=8.4 KiB, rowCount=199)
            +- Exchange (17)
               +- * Project (16)
                  +- * Filter (15)
                     +- * HashAggregate (14)
                        +- AQEShuffleRead (13)
                           +- ShuffleQueryStage (12), Statistics(sizeInBytes=17.0 KiB, rowCount=259)
                              +- Exchange (11)
                                 +- * HashAggregate (10)
                                    +- * HashAggregate (9)
                                       +- AQEShuffleRead (8)
                                          +- ShuffleQueryStage (7), Statistics(sizeInBytes=1067.6 KiB, rowCount=2.15E+4)
                                             +- Exchange (6)
                                                +- * HashAggregate (5)
                

== Physical Plan ==
AdaptiveSparkPlan (40)
+- == Final Plan ==
   ResultQueryStage (24)
   +- * Sort (23)
      +- AQEShuffleRead (22)
         +- ShuffleQueryStage (21), Statistics(sizeInBytes=344.0 B, rowCount=7)
            +- Exchange (20)
               +- * Project (19)
                  +- * Filter (18)
                     +- Window (17)
                        +- WindowGroupLimit (16)
                           +- * Sort (15)
                              +- AQEShuffleRead (14)
                                 +- ShuffleQueryStage (13), Statistics(sizeInBytes=688.0 B, rowCount=14)
                                    +- Exchange (12)
                                       +- WindowGroupLimit (11)
                                          +- * Sort (10)
                                             +- * HashAggregate (9)
                                                +- AQEShuffleRead (8)
                                                   +- ShuffleQueryStage (7), Statistics(siz

## Benchmark summary

In [6]:
summary = []
for query_name in QUERY_NAMES:
    disabled = aqe_disabled_results[query_name]
    enabled = aqe_enabled_results[query_name]
    summary.append({
        'query': query_name,
        'aqe_disabled_samples_seconds': [
            round(value, 3) for value in disabled['samples_seconds']
        ],
        'aqe_disabled_median_seconds': round(disabled['median_seconds'], 3),
        'aqe_enabled_samples_seconds': [
            round(value, 3) for value in enabled['samples_seconds']
        ],
        'aqe_enabled_median_seconds': round(enabled['median_seconds'], 3),
        'aqe_speedup_factor': round(
            disabled['median_seconds'] / enabled['median_seconds'],
            2,
        ),
    })

summary

[{'query': 'query_01_monthly_zone_demand',
  'aqe_disabled_samples_seconds': [3.676,
   3.246,
   1.946,
   1.938,
   2.105,
   1.696,
   1.799,
   1.574,
   1.577],
  'aqe_disabled_median_seconds': 1.938,
  'aqe_enabled_samples_seconds': [1.158,
   1.178,
   1.1,
   1.038,
   1.35,
   1.126,
   1.073,
   1.115,
   1.074],
  'aqe_enabled_median_seconds': 1.115,
  'aqe_speedup_factor': 1.74},
 {'query': 'query_02_average_trip_distance_by_weather',
  'aqe_disabled_samples_seconds': [2.355,
   2.184,
   1.937,
   1.771,
   2.287,
   2.038,
   1.886,
   2.169,
   1.775],
  'aqe_disabled_median_seconds': 2.038,
  'aqe_enabled_samples_seconds': [1.321,
   1.901,
   1.873,
   1.398,
   1.353,
   1.39,
   1.652,
   1.642,
   1.527],
  'aqe_enabled_median_seconds': 1.527,
  'aqe_speedup_factor': 1.33},
 {'query': 'query_03_air_quality_vs_demand',
  'aqe_disabled_samples_seconds': [1.865,
   1.477,
   2.122,
   1.611,
   1.854,
   1.566,
   1.265,
   1.287,
   1.237],
  'aqe_disabled_median_seco

In [7]:
spark.catalog.clearCache()
spark.stop()